# Pipeline de datos para tráfico IoT: corrida guiada

Este cuaderno ejecuta el pipeline sobre una captura **ya existente** y revisa sus salidas. Cubre:

1. Ingesta, extracción (NFStream), etiquetado y persistencia (`python -m pipeline run`).
2. Lectura de `manifest.json`, `traceability_report.json` y `flows.parquet`.
3. Comparación de tamaño y lectura Parquet (Snappy) frente al CSV baseline.
4. Handoff ML anti-fuga (`python -m pipeline preprocess-ml`).
5. PoC acotada: un detector no supervisado, ejecutado por el analista sobre el Parquet ya escrito.

**Alcance.** El pipeline es *downstream*: no captura tráfico, no inyecta eventos y no detecta ni clasifica como producto. La sección 5 es una prueba de concepto del analista, fuera del pipeline. Una sola corrida es evidencia preliminar y no una validación.

**Entrada requerida:** un PCAP y su JSON de eventos (contrato en `docs/CONTRATO-INTEGRACION.md`). Los PCAP no se versionan en el repositorio; ajuste las rutas en la celda siguiente.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Raíz del repositorio: primer directorio superior que contenga pyproject.toml.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

# --- Configuración (editar aquí) -------------------------------------------
PCAP_PATH = Path(os.environ.get("PIPELINE_PCAP", ROOT / "fixtures/upstream-run-00/capture.pcap"))
METADATA_PATH = Path(os.environ.get("PIPELINE_METADATA", ROOT / "fixtures/upstream-run-00/metadata.json"))
OUTPUT_DIR = Path(os.environ.get("PIPELINE_OUTPUT", ROOT / "output/notebook-run"))

RUN_PIPELINE = True   # False: reutiliza flows.parquet ya existente en OUTPUT_DIR
IDLE_TIMEOUT = 120    # s, valor por defecto del pipeline
ACTIVE_TIMEOUT = 1800 # s, valor por defecto del pipeline
N_DISSECTIONS = 20    # valor por defecto del pipeline
SPLIT_BY = "temporal" # temporal | episode | flow
SEED = 42
# ---------------------------------------------------------------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("python   :", sys.executable)
print("PCAP     :", PCAP_PATH)
print("metadata :", METADATA_PATH)
print("salida   :", OUTPUT_DIR)

In [ ]:
def run_cli(*args, ok_codes=(0,)):
    """Ejecuta `python -m pipeline ...` con el mismo intérprete del kernel."""
    cmd = [sys.executable, "-m", "pipeline", *map(str, args)]
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    log = (proc.stdout + proc.stderr).strip()
    print("\n".join(log.splitlines()[-25:]))
    if proc.returncode not in ok_codes:
        raise RuntimeError(f"El comando terminó con código {proc.returncode}")
    return proc.returncode


flows_path = OUTPUT_DIR / "flows.parquet"
if RUN_PIPELINE:
    missing = [p for p in (PCAP_PATH, METADATA_PATH) if not p.exists()]
    if missing:
        raise FileNotFoundError(
            f"Falta(n) {[str(p) for p in missing]}. Defina PIPELINE_PCAP / PIPELINE_METADATA "
            "o edite la celda de configuración (los PCAP no se versionan)."
        )
    run_cli(
        "run", "--pcap", PCAP_PATH, "--metadata", METADATA_PATH, "--output", OUTPUT_DIR,
        "--idle-timeout", IDLE_TIMEOUT, "--active-timeout", ACTIVE_TIMEOUT,
        "--n-dissections", N_DISSECTIONS, "--evaluate",
    )
elif not flows_path.exists():
    raise FileNotFoundError(f"RUN_PIPELINE=False pero no existe {flows_path}")

## 1. Manifiesto y correlación evento–flujo

In [ ]:
manifest = json.loads((OUTPUT_DIR / "manifest.json").read_text())
trace = json.loads((OUTPUT_DIR / "traceability_report.json").read_text())

print("Timeouts NFStream registrados:", manifest.get("nfstream_timeouts"))
print("Conteos de etiquetado       :", manifest.get("labeling_stats"))
print()
pd.Series(trace, name="valor").to_frame()

`background` es tráfico sin evento documentado en el JSON; no equivale a «benigno garantizado». Un evento sin flujo asociado aparece como huérfano (por ejemplo, si el JSON trae `duration_s` en 0: el pipeline aplica una ventana mínima de 1 s).

## 2. Flujos etiquetados

In [ ]:
flows = pd.read_parquet(flows_path)
print(f"{len(flows):,} flujos x {flows.shape[1]} columnas")

counts = flows["flow_label"].value_counts()
display(counts.to_frame("flujos"))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
counts.plot.bar(ax=ax[0], color=["#888", "#c0392b", "#2980b9"][: len(counts)])
ax[0].set_title("Flujos por etiqueta"); ax[0].set_xlabel(""); ax[0].tick_params(axis="x", rotation=0)

t0 = flows["bidirectional_first_seen_ms"].min()
flows["_t_s"] = (flows["bidirectional_first_seen_ms"] - t0) / 1000
for lab, g in flows.groupby("flow_label"):
    ax[1].scatter(g["_t_s"], np.full(len(g), lab), s=6, alpha=0.4)
ax[1].set_title("Inicio de flujos en el tiempo"); ax[1].set_xlabel("segundos desde el primer flujo")
plt.tight_layout(); plt.show()

In [ ]:
cols = ["src_ip", "dst_ip", "protocol", "application_name", "bidirectional_packets",
        "bidirectional_bytes", "flow_label", "event_id"]
flows[[c for c in cols if c in flows.columns]].head(10)

## 3. Parquet (Snappy) frente a CSV baseline

In [ ]:
csv_path = OUTPUT_DIR / "flows_baseline.csv"
io_path = OUTPUT_DIR / "io_metrics.json"
csv_mb = csv_path.stat().st_size / 1e6
pq_mb = flows_path.stat().st_size / 1e6
print(f"CSV     : {csv_mb:8.3f} MB")
print(f"Parquet : {pq_mb:8.3f} MB   (razón CSV/Parquet = {csv_mb / pq_mb:.2f}x)")

if io_path.exists():
    io = json.loads(io_path.read_text())
    print(f"Lectura (benchmark del pipeline): CSV {io['csv_read_seconds']} s, "
          f"Parquet {io['parquet_read_seconds']} s -> {io['read_speedup']}x")
    q = pd.DataFrame(io.get("analytical_queries", []))
    if not q.empty:
        display(q[["name", "csv_seconds", "parquet_seconds", "speedup", "result_rows"]])
print("\nNota: los tiempos de una corrida pequeña son ruidosos; las cifras de la memoria provienen de la corrida piloto.")

## 4. Handoff ML anti-fuga (`preprocess-ml`)

Particiona sin barajar al azar, ajusta imputación, escalado y One-Hot **solo con train**, y excluye identificadores, IP, puertos, timestamps absolutos y columnas derivadas de la etiqueta. Si la partición deja una clase sin muestras en test, o no supera una validación, queda registrado en `validation_report.json`.

In [ ]:
ML_DIR = OUTPUT_DIR / "ml-handoff"
# Código 8 = la validación anti-fuga reportó un fallo; los archivos igual se escriben y se revisan abajo.
rc = run_cli("preprocess-ml", "--input", flows_path, "--output", ML_DIR,
             "--split-by", SPLIT_BY, "--seed", SEED, ok_codes=(0, 8))
if rc == 8:
    print("\n[AVISO] validate_ml reportó al menos un chequeo fallido; ver la tabla de validación.")

train = pd.read_parquet(ML_DIR / "train_processed.parquet")
test = pd.read_parquet(ML_DIR / "test_processed.parquet")
fm = json.loads((ML_DIR / "feature_manifest.json").read_text())
print(f"\ntrain={len(train):,}  test={len(test):,}  features={fm['n_features_out']}")
display(pd.DataFrame({"train": train["flow_label"].value_counts(),
                      "test": test["flow_label"].value_counts()}).fillna(0).astype(int))

In [ ]:
report = json.loads((ML_DIR / "validation_report.json").read_text())
print({k: v for k, v in report.items() if not isinstance(v, (list, dict))})
checks = report.get("checks")
if isinstance(checks, list):
    display(pd.DataFrame(checks))
elif isinstance(checks, dict):
    display(pd.Series(checks).to_frame("resultado"))

## 5. PoC acotada: detector no supervisado (ejecutada por el analista)

Un `IsolationForest` se ajusta **solo con tráfico no etiquetado como ataque de la partición train** (`background` y `benign`, como aproximación a tráfico normal reciente) y puntúa el test. No es un IDS ni un entrenamiento formal; su único propósito es comprobar que el Parquet entregado se puede consumir de punta a punta. Con una corrida única, el resultado es evidencia preliminar.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

feat = [c for c in train.columns if c != "flow_label"]
X_train = train.loc[train["flow_label"] != "attack", feat]
X_test, y_test = test[feat], (test["flow_label"] == "attack").astype(int)

print(f"ajuste con {len(X_train):,} flujos no-ataque; test: {int(y_test.sum())} ataque / {int((1 - y_test).sum())} otros")
if len(X_train) < 10 or y_test.nunique() < 2:
    print("Partición insuficiente para evaluar (falta una clase en test o train casi vacío). "
          "Pruebe SPLIT_BY='episode' o una captura más larga.")
else:
    model = IsolationForest(n_estimators=200, random_state=SEED, n_jobs=1).fit(X_train)
    score = -model.score_samples(X_test)          # mayor = más anómalo
    thr = np.quantile(-model.score_samples(X_train), 0.95)
    pred = (score > thr).astype(int)
    print(f"ROC-AUC (ataque vs resto): {roc_auc_score(y_test, score):.3f}")
    print("Umbral = percentil 95 de las puntuaciones de train")
    print(confusion_matrix(y_test, pred, labels=[0, 1]))
    print(classification_report(y_test, pred, target_names=["no-ataque", "ataque"], zero_division=0))

## Cierre

Reproducibilidad: la corrida queda descrita por `manifest.json` (parámetros, conteos, timeouts) y por la semilla de `preprocess-ml`. Para repetirla en otra máquina, basta con el mismo PCAP, el mismo JSON y los parámetros de la celda de configuración.